# Laya airline satisfaction

Kaggle settings: Internet On, accelerator GPU T4 x2, and the competition data attached. This notebook uses one T4. The second T4 is left unused because dual-GPU training would need a generated Python file.

Both notebooks use seed 42, 50,000 fine-tune rows, 5,000 validation rows, and the same `satisfied` / `not_satisfied` question. Upload `submission_laya.csv` only if its validation ROC AUC beats `submission_kev.csv`. Within 0.005 AUC, choose the lower Brier score.

MLflow logs to `sqlite:///mlflow.db` at the repo root when this notebook runs locally. On Kaggle it logs to `/kaggle/working/mlflow.db` and writes `mlflow_run_summary_laya.json`. Run the last cell locally after downloading that file.


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlflow", "scikit-learn", "numpy"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NandhaKishorM/laya.git"])


In [ ]:
import csv
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

import numpy as np

SEED = 42
N_TRAIN = 50000
N_VAL = 5000
EPOCHS = 2
EXPERIMENT = "s6e10-satisfaction"
QUESTION_ID = "satisfaction"
INSTRUCTIONS = "Is this passenger satisfied with the airline?"
CRITERIA = {
    "satisfied": "The passenger is satisfied.",
    "not_satisfied": "The passenger is neutral or dissatisfied.",
}
FEATURE_COLUMNS = [
    "Age",
    "Flight Distance",
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
]
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset" / "playground-series-s6e10").is_dir() or (candidate / "notebooks").is_dir():
            return candidate
    return here


ROOT = repo_root()
ON_KAGGLE = Path("/kaggle/working").is_dir()
WORK = Path("/kaggle/working") if ON_KAGGLE else ROOT / "artifacts"
WORK.mkdir(parents=True, exist_ok=True)


def find_competition_file(filename):
    seeds = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        seeds.append(kaggle_input)
    seen = set()
    for root in seeds:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        direct = root / "dataset" / "playground-series-s6e10" / filename
        if direct.is_file():
            return direct
        if str(root).startswith("/kaggle/input"):
            matches = sorted(path for path in root.rglob(filename) if path.is_file())
            if matches:
                return matches[0]
    raise FileNotFoundError(filename)


def cell_text(column, value):
    if value is None:
        return "missing"
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return "missing"
    if text.endswith(".0") and text[:-2].lstrip("-").isdigit():
        return text[:-2]
    return text


def state_text(row):
    return "\n".join(f"{column}: {cell_text(column, row[column])}" for column in FEATURE_COLUMNS)


def is_satisfied(value):
    return str(value).strip().lower() in {"true", "1", "yes", "satisfied"}


def read_table(path):
    with open(path, newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))


def stratified_split(ids_true, ids_false, n_val, n_train, seed):
    rng = np.random.RandomState(seed)
    ids_true = np.array(ids_true, dtype=object)
    ids_false = np.array(ids_false, dtype=object)
    rng.shuffle(ids_true)
    rng.shuffle(ids_false)
    total = len(ids_true) + len(ids_false)
    n_val_true = min(int(round(n_val * len(ids_true) / total)), len(ids_true), n_val)
    n_val_false = n_val - n_val_true
    if n_val_false > len(ids_false):
        n_val_false = len(ids_false)
        n_val_true = n_val - n_val_false
    val_ids = np.concatenate([ids_true[:n_val_true], ids_false[:n_val_false]])
    rest_true = ids_true[n_val_true:]
    rest_false = ids_false[n_val_false:]
    rest_total = len(rest_true) + len(rest_false)
    n_train_true = min(int(round(n_train * len(rest_true) / rest_total)), len(rest_true), n_train)
    n_train_false = n_train - n_train_true
    if n_train_false > len(rest_false):
        n_train_false = len(rest_false)
        n_train_true = n_train - n_train_false
    train_ids = np.concatenate([rest_true[:n_train_true], rest_false[:n_train_false]])
    return [str(item) for item in train_ids], [str(item) for item in val_ids]


def fingerprint(ids):
    return hashlib.sha256(",".join(sorted(ids)).encode()).hexdigest()


def question_body():
    return {
        QUESTION_ID: {
            "type": "choice",
            "instructions": INSTRUCTIONS,
            "criteria": dict(CRITERIA),
        }
    }


def label_for(row):
    return "satisfied" if is_satisfied(row["satisfaction"]) else "not_satisfied"


def competition_metrics(y_true, probabilities):
    from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1.0 - 1e-6)
    return {
        "val_roc_auc": float(roc_auc_score(y, p)),
        "val_brier": float(brier_score_loss(y, p)),
        "val_log_loss": float(log_loss(y, p)),
    }


def satisfied_probability(payload):
    answers = payload["answers"] if isinstance(payload, dict) and "answers" in payload else payload
    answer = answers[QUESTION_ID]
    return float(answer["probabilities"]["satisfied"])


def require_cuda():
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Turn on Kaggle accelerator GPU T4 x2 and Internet, then run the notebook again.")
    props = torch.cuda.get_device_properties(0)
    print(torch.cuda.get_device_name(0), round(props.total_memory / 1e9, 2), "GB")


def write_submission(path, ids, probabilities):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "satisfaction"])
        for row_id, probability in zip(ids, probabilities):
            writer.writerow([row_id, f"{float(probability):.10f}"])
    return path


class Tracker:
    def __init__(self, run_name, summary_name):
        import mlflow

        self.mlflow = mlflow
        self.run_name = run_name
        self.summary_name = summary_name
        self.params = {}
        self.metrics = []
        self.tags = {"model": run_name}
        self.uri = "sqlite:////kaggle/working/mlflow.db" if ON_KAGGLE else "sqlite:///" + str(ROOT / "mlflow.db")
        mlflow.set_tracking_uri(self.uri)
        mlflow.set_experiment(EXPERIMENT)
        mlflow.end_run()
        self.run = mlflow.start_run(run_name=run_name)
        mlflow.set_tags(self.tags)

    def log_params(self, params):
        clean = {key: "" if value is None else value if isinstance(value, (int, float, str)) else str(value) for key, value in params.items()}
        self.params.update(clean)
        self.mlflow.log_params(clean)

    def log_metrics(self, metrics, step):
        for key, value in metrics.items():
            numeric = float(value)
            self.mlflow.log_metric(key, numeric, step=int(step))
            self.metrics.append({"key": key, "value": numeric, "step": int(step)})

    def set_tag(self, key, value):
        self.tags[key] = str(value)
        self.mlflow.set_tag(key, str(value))

    def finish(self):
        if getattr(self, "closed", False):
            return
        self.closed = True
        self.mlflow.end_run()
        summary = {
            "source": "kaggle" if ON_KAGGLE else "local",
            "run_name": self.run_name,
            "params": self.params,
            "metrics": self.metrics,
            "tags": self.tags,
        }
        for folder in {WORK, Path.cwd()}:
            destination = folder / self.summary_name
            destination.write_text(json.dumps(summary, indent=2), encoding="utf-8")
            print("wrote", destination)

SUMMARY_NAME = 'mlflow_run_summary_laya.json'


In [ ]:
train_rows = read_table(find_competition_file("train.csv"))
test_rows = read_table(find_competition_file("test.csv"))
by_id = {row["id"]: row for row in train_rows}
true_ids = [row["id"] for row in train_rows if is_satisfied(row["satisfaction"])]
false_ids = [row["id"] for row in train_rows if not is_satisfied(row["satisfaction"])]
train_ids, val_ids = stratified_split(true_ids, false_ids, N_VAL, N_TRAIN, SEED)
if set(train_ids) & set(val_ids):
    raise RuntimeError("train and validation ids overlap")
print("train_rows", len(train_rows), "test_rows", len(test_rows))
print("finetune", len(train_ids), "validation", len(val_ids))
print("satisfied_rate_train", sum(is_satisfied(by_id[row_id]["satisfaction"]) for row_id in train_ids) / len(train_ids))
print("satisfied_rate_val", sum(is_satisfied(by_id[row_id]["satisfaction"]) for row_id in val_ids) / len(val_ids))
print("train_fingerprint", fingerprint(train_ids))
print("val_fingerprint", fingerprint(val_ids))
print(state_text(by_id[val_ids[0]]))


## Train and score

Zero-shot and each fine-tune epoch are scored on the same 5,000 validation rows. The test file is scored only by the better of those runs. `soft-ce` is Laya's documented single-device loss; it beat the default RLCD objective on Laya's own typed-decision benchmark and uses less memory on one T4.


In [ ]:
import laya
from laya.train import TrainConfig, load_checkpoint, items_from_rows, save_checkpoint, train_model

require_cuda()
tracker = Tracker("laya", SUMMARY_NAME)
import atexit
atexit.register(tracker.finish)
tracker.log_params({
    "model_id": "convaiinnovations/laya",
    "seed": SEED,
    "n_finetune": len(train_ids),
    "n_val": len(val_ids),
    "epochs": EPOCHS,
    "learning_rate": 2.5e-5,
    "head_learning_rate": 1e-4,
    "dtype": "fp16",
    "batch_size": 4,
    "grad_accum": 8,
    "loss": "soft-ce",
    "question_id": QUESTION_ID,
    "instructions": INSTRUCTIONS,
    "train_fingerprint": fingerprint(train_ids),
    "val_fingerprint": fingerprint(val_ids),
})

val_states = [state_text(by_id[row_id]) for row_id in val_ids]
val_y = [1 if is_satisfied(by_id[row_id]["satisfaction"]) else 0 for row_id in val_ids]
questions = question_body()


def score_laya(agent, states):
    started = time.perf_counter()
    probabilities = []
    for index, state in enumerate(states, start=1):
        probabilities.append(satisfied_probability(agent.predict(state, questions)))
        if index % 500 == 0 or index == len(states):
            print(f"scored {index}/{len(states)}", flush=True)
    return probabilities, (time.perf_counter() - started) / len(states)


def load_laya(source):
    if Path(source).is_dir():
        return laya.Agent(str(source), device="cuda")
    try:
        return laya.load(source, device="cuda")
    except TypeError:
        return laya.load(source)


def release(obj):
    del obj
    gc.collect()
    import torch
    torch.cuda.empty_cache()


print("zero-shot")
agent = load_laya("convaiinnovations/laya")
zero_probabilities, zero_seconds = score_laya(agent, val_states)
release(agent)
zero_metrics = competition_metrics(val_y, zero_probabilities)
zero_metrics["sec_per_row"] = zero_seconds
tracker.log_metrics(zero_metrics, step=0)
print("zero-shot", zero_metrics)

laya_rows = []
for row_id in train_ids:
    row = by_id[row_id]
    laya_rows.append({
        "state": state_text(row),
        "questions": question_body(),
        "expected": {QUESTION_ID: label_for(row)},
    })
train_path = WORK / "laya_train.jsonl"
train_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in laya_rows), encoding="utf-8")

epoch_metrics = []
epoch_dirs = []
source = "convaiinnovations/laya"
for epoch in range(1, EPOCHS + 1):
    model, tokenizer, config = load_checkpoint(source)
    max_len = config.get("max_len", 512)
    head_max_len = config.get("head_max_len", 192)
    items, skipped = items_from_rows(tokenizer, laya_rows, max_len, head_max_len)
    if not items:
        raise RuntimeError(f"Laya produced no training items: {skipped}")
    print("epoch", epoch, "items", len(items), "skipped", skipped)
    train_model(
        model,
        tokenizer,
        items,
        TrainConfig(epochs=1, micro_batch=4, grad_accum=8, loss="soft-ce", seed=SEED, log_every=50),
        "cuda",
        max_len,
        head_max_len,
    )
    epoch_dir = WORK / f"laya-epoch-{epoch}"
    if epoch_dir.exists():
        shutil.rmtree(epoch_dir)
    save_checkpoint(model, tokenizer, config, str(epoch_dir))
    release(model)
    agent = load_laya(epoch_dir)
    probabilities, seconds = score_laya(agent, val_states)
    release(agent)
    metrics = competition_metrics(val_y, probabilities)
    metrics["sec_per_row"] = seconds
    tracker.log_metrics(metrics, step=epoch)
    print(f"epoch {epoch}", metrics)
    epoch_metrics.append(metrics)
    epoch_dirs.append(epoch_dir)
    source = str(epoch_dir)

best_step = 0
best_metrics = zero_metrics
best_source = "convaiinnovations/laya"
for epoch, metrics in enumerate(epoch_metrics, start=1):
    if metrics["val_roc_auc"] > best_metrics["val_roc_auc"]:
        best_step = epoch
        best_metrics = metrics
        best_source = str(epoch_dirs[epoch - 1])
tracker.set_tag("chosen", "zero-shot" if best_step == 0 else f"epoch-{best_step}")
print("chosen", tracker.tags["chosen"], best_metrics)

print("scoring test with", best_source)
agent = load_laya(best_source)
test_states = [state_text(row) for row in test_rows]
test_probabilities, test_seconds = score_laya(agent, test_states)
release(agent)
submission = write_submission(WORK / "submission_laya.csv", [row["id"] for row in test_rows], test_probabilities)
if Path.cwd() != WORK:
    write_submission(Path.cwd() / "submission_laya.csv", [row["id"] for row in test_rows], test_probabilities)
print("RESULT laya", "chosen", tracker.tags["chosen"], "val_roc_auc", best_metrics["val_roc_auc"], "val_brier", best_metrics["val_brier"], "sec_per_row", best_metrics["sec_per_row"], "test_sec_per_row", test_seconds)
print("submission", submission)
print("Higher validation ROC AUC wins. Within 0.005, the lower Brier score wins.")
tracker.finish()


## Import the Kaggle run into local MLflow

Skip this cell on Kaggle. After the run finishes, download `mlflow_run_summary_laya.json` into this repo and run this cell on your machine. It writes the same metrics into `mlflow.db`.


In [ ]:
import hashlib
import json
from pathlib import Path

summary_name = 'mlflow_run_summary_laya.json'

def repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset" / "playground-series-s6e10").is_dir() or (candidate / "notebooks").is_dir():
            return candidate
    return here

candidates = [
    Path.cwd() / summary_name,
    Path.cwd() / "artifacts" / summary_name,
    repo_root() / summary_name,
    repo_root() / "artifacts" / summary_name,
    Path("/kaggle/working") / summary_name,
]
summary_path = next((path for path in candidates if path.is_file()), None)
if summary_path is None:
    print("No Kaggle summary found. Nothing to import.")
elif Path("/kaggle/working").is_dir():
    print("Still on Kaggle. Download", summary_name, "and run this cell from the repo.")
else:
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary.get("source") != "kaggle":
        print("This run was already logged locally.")
    else:
        import mlflow

        digest = hashlib.sha256(json.dumps(summary["metrics"], sort_keys=True).encode()).hexdigest()[:16]
        uri = "sqlite:///" + str(repo_root() / "mlflow.db")
        mlflow.set_tracking_uri(uri)
        experiment = mlflow.set_experiment("s6e10-satisfaction")
        existing = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string=f"tags.summary_sha = '{digest}'",
        )
        if existing.shape[0]:
            print("Already imported", digest)
        else:
            with mlflow.start_run(run_name=summary["run_name"]):
                mlflow.set_tags({**summary.get("tags", {}), "summary_sha": digest, "imported_from": "kaggle"})
                if summary.get("params"):
                    mlflow.log_params(summary["params"])
                for item in summary["metrics"]:
                    mlflow.log_metric(item["key"], item["value"], step=item["step"])
            print("Imported", summary["run_name"], "into", uri)
            print("Compare with: mlflow ui --backend-store-uri", uri)
